In [1]:
!pip -q install -U transformers datasets evaluate accelerate sentencepiece rouge_score sacrebleu safetensors

from google.colab import drive
drive.mount('/content/drive')

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 138.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 46.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 56.1 MB/s eta 0:00:00
Mounted at /content/drive


In [2]:
import os
import gc
import json
import math
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# ============================================================
# Reproducibility
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ============================================================
# Project paths
# ============================================================

PROJECT_DIR = Path("/content/drive/MyDrive/Senior2_Medical_Captioning")

# Expected master table path
MASTER_PATH = PROJECT_DIR / "master_caption_concepts_scored.csv"

# Output directory for safe 5-epoch Experiment 3
OUTPUT_DIR = PROJECT_DIR / "models" / "exp3_pred_micro_terms_t5_base_safe5"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Debug mode:
# False = official run
# True  = quick test on smaller data
FAST_DEBUG = False

VALID_SAMPLE_SIZE = 3000
TRAIN_DEBUG_SIZE = 20000
VALID_DEBUG_SIZE = 1000

print("PROJECT_DIR:", PROJECT_DIR)
print("MASTER_PATH:", MASTER_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("FAST_DEBUG:", FAST_DEBUG)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU: CPU only")

PROJECT_DIR: /content/drive/MyDrive/Senior2_Medical_Captioning
MASTER_PATH: /content/drive/MyDrive/Senior2_Medical_Captioning/master_caption_concepts_scored.csv
OUTPUT_DIR: /content/drive/MyDrive/Senior2_Medical_Captioning/models/exp3_pred_micro_terms_t5_base_safe5
FAST_DEBUG: False
GPU: NVIDIA A100-SXM4-80GB


In [3]:
# ============================================================
# Helper functions for file discovery
# ============================================================

def find_csv_candidates(root_dir):
    patterns = [
        "*master*.csv",
        "*caption*concept*.csv",
        "*scored*.csv",
        "*.csv",
    ]

    candidates = []
    for pattern in patterns:
        candidates.extend(list(root_dir.rglob(pattern)))

    candidates = sorted(set(candidates))
    return candidates


def find_label_vocab_candidates(root_dir):
    patterns = [
        "*label_vocab*.json",
        "*vocab*.json",
        "*labels*.json",
    ]

    candidates = []
    for pattern in patterns:
        candidates.extend(list(root_dir.rglob(pattern)))

    candidates = sorted(set(candidates))
    return candidates


# ============================================================
# Load master table
# ============================================================

if not MASTER_PATH.exists():
    print("MASTER_PATH not found. Searching inside PROJECT_DIR...")
    csv_candidates = find_csv_candidates(PROJECT_DIR)

    print("CSV candidates:")
    for p in csv_candidates[:30]:
        print(" -", p)

    if len(csv_candidates) == 0:
        raise FileNotFoundError(
            "No CSV files found. Please check PROJECT_DIR or upload/build the master table."
        )

    # Prefer files that contain master/scored in the name
    preferred = [
        p for p in csv_candidates
        if ("master" in p.name.lower() or "scored" in p.name.lower())
    ]

    MASTER_PATH = preferred[0] if len(preferred) > 0 else csv_candidates[0]

print("Using MASTER_PATH:", MASTER_PATH)

master_df = pd.read_csv(MASTER_PATH)

print("Master shape:", master_df.shape)
print("Master columns:")
print(master_df.columns.tolist())

display(master_df.head())


# ============================================================
# Load label vocabulary
# ============================================================

LABEL_VOCAB_PATHS = [
    Path("/content/drive/MyDrive/ImageCLEF/artifacts_consultive_swin/label_vocab.json"),
    PROJECT_DIR / "label_vocab.json",
]

LABEL_VOCAB_PATH = None

for p in LABEL_VOCAB_PATHS:
    if p.exists():
        LABEL_VOCAB_PATH = p
        break

if LABEL_VOCAB_PATH is None:
    print("\nLabel vocab not found in expected paths. Searching...")
    label_candidates = find_label_vocab_candidates(Path("/content/drive/MyDrive"))

    print("Label vocab candidates:")
    for p in label_candidates[:30]:
        print(" -", p)

    if len(label_candidates) == 0:
        raise FileNotFoundError(
            "No label_vocab JSON found. This file is important to keep CUI token order consistent."
        )

    # Prefer exact label_vocab name
    preferred = [p for p in label_candidates if "label_vocab" in p.name.lower()]
    LABEL_VOCAB_PATH = preferred[0] if len(preferred) > 0 else label_candidates[0]

print("\nUsing LABEL_VOCAB_PATH:", LABEL_VOCAB_PATH)

with open(LABEL_VOCAB_PATH, "r", encoding="utf-8") as f:
    label_obj = json.load(f)

if isinstance(label_obj, dict) and "labels" in label_obj:
    cui_list = label_obj["labels"]
elif isinstance(label_obj, list):
    cui_list = label_obj
else:
    raise ValueError(
        "Unsupported label vocab structure. Expected dict with key 'labels' or a list of CUIs."
    )

cui_list = [str(c) for c in cui_list]

print("Number of CUIs:", len(cui_list))
print("First 10 CUIs:", cui_list[:10])

Using MASTER_PATH: /content/drive/MyDrive/Senior2_Medical_Captioning/master_caption_concepts_scored.csv
Master shape: (116604, 20)
Master columns:
['ID', 'reference_caption', 'License', 'Attribution', 'gt_CUIs', 'pred_CUIs_micro', 'pred_CUIs_coverage', 'split', 'image_filename', 'caption_image_zip_path', 'concept_image_zip_path', 'gt_CUIs_count', 'pred_CUIs_micro_count', 'pred_CUIs_coverage_count', 'pred_CUIs_micro_precision', 'pred_CUIs_micro_recall', 'pred_CUIs_micro_f1', 'pred_CUIs_coverage_precision', 'pred_CUIs_coverage_recall', 'pred_CUIs_coverage_f1']


,ID,reference_caption,License,Attribution,gt_CUIs,pred_CUIs_micro,pred_CUIs_coverage,split,image_filename,caption_image_zip_path,concept_image_zip_path,gt_CUIs_count,pred_CUIs_micro_count,pred_CUIs_coverage_count,pred_CUIs_micro_precision,pred_CUIs_micro_recall,pred_CUIs_micro_f1,pred_CUIs_coverage_precision,pred_CUIs_coverage_recall,pred_CUIs_coverage_f1
0,ImageCLEFmedical_Caption_2026_train_0,Head CT demonstrating left parotiditis.,CC BY,Peres et al.,C0040405,C0040405,C0040405,train,ImageCLEFmedical_Caption_2026_train_0.jpg,dev_caption/images/ImageCLEFmedical_Caption_20...,dev_concept/images/ImageCLEFmedical_Caption_20...,1,1,1,1.0,1.0,1.000000,1.0,1.0,1.000000
1,ImageCLEFmedical_Caption_2026_train_1,Chest X-ray showing enlarged cardiac silhouett...,CC BY-NC,Al Mulhim et al.,C1306645;C0817096;C0442800;C0018787;C0242073,C0817096;C1306645,C0817096;C1306645,train,ImageCLEFmedical_Caption_2026_train_1.jpg,dev_caption/images/ImageCLEFmedical_Caption_20...,dev_concept/images/ImageCLEFmedical_Caption_20...,5,2,2,1.0,0.4,0.571429,1.0,0.4,0.571429
2,ImageCLEFmedical_Caption_2026_train_2,CT chest axial view showing a huge ascending a...,CC BY-NC,Al Mulhim et al.,C0040405;C0856747,C0040405,C0040405,train,ImageCLEFmedical_Caption_2026_train_2.jpg,dev_caption/images/ImageCLEFmedical_Caption_20...,dev_concept/images/ImageCLEFmedical_Caption_20...,2,1,1,1.0,0.5,0.666667,1.0,0.5,0.666667
3,ImageCLEFmedical_Caption_2026_train_3,Acquired renal cysts in end-stage renal failur...,CC BY,Vester et al.,C0041618,C0041618,C0041618,train,ImageCLEFmedical_Caption_2026_train_3.jpg,dev_caption/images/ImageCLEFmedical_Caption_20...,dev_concept/images/ImageCLEFmedical_Caption_20...,1,1,1,1.0,1.0,1.000000,1.0,1.0,1.000000
4,ImageCLEFmedical_Caption_2026_train_4,Computed tomography (CT) shows floating thromb...,CC BY,Sato et al.,C0040405;C0040053,C0040405,C0040405,train,ImageCLEFmedical_Caption_2026_train_4.jpg,dev_caption/images/ImageCLEFmedical_Caption_20...,dev_concept/images/ImageCLEFmedical_Caption_20...,2,1,1,1.0,0.5,0.666667,1.0,0.5,0.666667



Using LABEL_VOCAB_PATH: /content/drive/MyDrive/ImageCLEF/artifacts_consultive_swin/label_vocab.json
Number of CUIs: 2646
First 10 CUIs: ['C0000726', 'C0000741', 'C0000833', 'C0000846', 'C0000962', 'C0001074', 'C0001080', 'C0001162', 'C0001168', 'C0001208']


In [4]:
# ============================================================
# Normalize master table columns to standard names
# Required standard columns:
# ID, split, reference_caption, gt_CUIs, pred_CUIs_micro, pred_CUIs_coverage
# ============================================================

def pick_column(df, candidates, target_name, required=True):
    lower_map = {c.lower(): c for c in df.columns}

    for cand in candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]

    # Try partial matching
    for col in df.columns:
        col_l = col.lower()
        for cand in candidates:
            cand_l = cand.lower()
            if cand_l in col_l:
                return col

    if required:
        raise ValueError(
            f"Could not find column for '{target_name}'. "
            f"Candidates tried: {candidates}\n"
            f"Available columns: {df.columns.tolist()}"
        )

    return None


id_col = pick_column(
    master_df,
    ["ID", "image_id", "imageid", "ImageID", "sample_id"],
    "ID"
)

split_col = pick_column(
    master_df,
    ["split", "Split", "data_split", "subset"],
    "split"
)

caption_col = pick_column(
    master_df,
    ["reference_caption", "caption", "Caption", "text", "target_text", "reference"],
    "reference_caption"
)

gt_col = pick_column(
    master_df,
    ["gt_CUIs", "gt_cuis", "GT_CUIs", "ground_truth_cuis", "gt_concepts", "concepts"],
    "gt_CUIs"
)

pred_micro_col = pick_column(
    master_df,
    ["pred_CUIs_micro", "pred_cuis_micro", "pred_micro_cuis", "micro_CUIs", "micro_cuis"],
    "pred_CUIs_micro"
)

pred_coverage_col = pick_column(
    master_df,
    ["pred_CUIs_coverage", "pred_cuis_coverage", "pred_coverage_cuis", "coverage_CUIs", "coverage_cuis"],
    "pred_CUIs_coverage"
)

print("Detected columns:")
print("ID:", id_col)
print("split:", split_col)
print("reference_caption:", caption_col)
print("gt_CUIs:", gt_col)
print("pred_CUIs_micro:", pred_micro_col)
print("pred_CUIs_coverage:", pred_coverage_col)

# Create standardized dataframe
std_df = master_df.copy()

std_df["ID"] = std_df[id_col].astype(str)
std_df["split"] = std_df[split_col].astype(str).str.lower().str.strip()
std_df["reference_caption"] = std_df[caption_col].astype(str)
std_df["gt_CUIs"] = std_df[gt_col].fillna("").astype(str)
std_df["pred_CUIs_micro"] = std_df[pred_micro_col].fillna("").astype(str)
std_df["pred_CUIs_coverage"] = std_df[pred_coverage_col].fillna("").astype(str)

# Normalize split names
split_map = {
    "val": "valid",
    "validation": "valid",
    "dev": "valid",
    "train": "train",
    "training": "train",
    "test": "test"
}

std_df["split"] = std_df["split"].replace(split_map)

print("\nSplit counts:")
print(std_df["split"].value_counts())

required_splits = {"train", "valid"}
existing_splits = set(std_df["split"].unique())

missing_splits = required_splits - existing_splits
if missing_splits:
    raise ValueError(
        f"Missing required splits: {missing_splits}. "
        f"Existing splits: {sorted(existing_splits)}"
    )

# Keep standardized df
master_df = std_df

display(master_df[[
    "ID",
    "split",
    "reference_caption",
    "gt_CUIs",
    "pred_CUIs_micro",
    "pred_CUIs_coverage"
]].head())

Detected columns:
ID: ID
split: split
reference_caption: reference_caption
gt_CUIs: gt_CUIs
pred_CUIs_micro: pred_CUIs_micro
pred_CUIs_coverage: pred_CUIs_coverage

Split counts:
split
train    97364
valid    19240
Name: count, dtype: int64


,ID,split,reference_caption,gt_CUIs,pred_CUIs_micro,pred_CUIs_coverage
0,ImageCLEFmedical_Caption_2026_train_0,train,Head CT demonstrating left parotiditis.,C0040405,C0040405,C0040405
1,ImageCLEFmedical_Caption_2026_train_1,train,Chest X-ray showing enlarged cardiac silhouett...,C1306645;C0817096;C0442800;C0018787;C0242073,C0817096;C1306645,C0817096;C1306645
2,ImageCLEFmedical_Caption_2026_train_2,train,CT chest axial view showing a huge ascending a...,C0040405;C0856747,C0040405,C0040405
3,ImageCLEFmedical_Caption_2026_train_3,train,Acquired renal cysts in end-stage renal failur...,C0041618,C0041618,C0041618
4,ImageCLEFmedical_Caption_2026_train_4,train,Computed tomography (CT) shows floating thromb...,C0040405;C0040053,C0040405,C0040405


In [6]:
# ============================================================
# Cell 5 - Corrected CUI-to-UMLS-Term Mapping Loader
# ============================================================

import re
import pandas as pd
import json
from pathlib import Path

# We explicitly prefer the real mapping file, not prepared train/valid datasets.
MAPPING_PATH = PROJECT_DIR / "cui_to_umls_terms.csv"

if not MAPPING_PATH.exists():
    raise FileNotFoundError(
        f"Expected mapping file not found:\n{MAPPING_PATH}\n\n"
        "Please check that cui_to_umls_terms.csv exists inside Senior2_Medical_Captioning."
    )

print("Using mapping file:", MAPPING_PATH)

mapping_df = pd.read_csv(MAPPING_PATH)

print("\nMapping shape:", mapping_df.shape)
print("Mapping columns:")
print(mapping_df.columns.tolist())

display(mapping_df.head())


# ============================================================
# Detect CUI and term columns robustly
# ============================================================

def detect_cui_column(df):
    """
    Detect the column that contains CUI codes such as C0040405.
    """
    # First try common column names
    common_cui_names = [
        "cui", "CUI", "umls_cui", "UMLS_CUI",
        "concept_id", "Concept_ID", "concept", "Concept"
    ]

    for name in common_cui_names:
        if name in df.columns:
            return name

    # Otherwise detect by content pattern C + 7 digits
    cui_pattern = re.compile(r"^C\d{7}$")

    best_col = None
    best_count = 0

    for col in df.columns:
        values = df[col].dropna().astype(str).head(200)
        count = sum(bool(cui_pattern.match(v.strip())) for v in values)

        if count > best_count:
            best_count = count
            best_col = col

    if best_col is not None and best_count > 0:
        return best_col

    raise ValueError(
        "Could not detect CUI column. "
        f"Available columns: {df.columns.tolist()}"
    )


def detect_term_column(df, cui_col):
    """
    Detect the column that contains readable UMLS terms.
    """
    # First try common term column names
    common_term_names = [
        "term", "Term",
        "name", "Name",
        "preferred_name", "Preferred_Name",
        "preferred_term", "Preferred_Term",
        "umls_term", "UMLS_Term",
        "label", "Label",
        "str", "STR"
    ]

    for name in common_term_names:
        if name in df.columns and name != cui_col:
            return name

    # Otherwise pick the best text-like column that is not the CUI column
    candidate_cols = [c for c in df.columns if c != cui_col]

    best_col = None
    best_score = -1

    for col in candidate_cols:
        values = df[col].dropna().astype(str).head(200)

        if len(values) == 0:
            continue

        # Score: average length + number of alphabetic-containing values
        avg_len = values.str.len().mean()
        alpha_count = values.str.contains(r"[A-Za-z]", regex=True).sum()
        score = avg_len + alpha_count

        if score > best_score:
            best_score = score
            best_col = col

    if best_col is not None:
        return best_col

    raise ValueError(
        "Could not detect UMLS term column. "
        f"Available columns: {df.columns.tolist()}"
    )


cui_col = detect_cui_column(mapping_df)
term_col = detect_term_column(mapping_df, cui_col)

print("\nDetected CUI column:", cui_col)
print("Detected term column:", term_col)


# ============================================================
# Build mapping dictionary
# ============================================================

mapping_df[cui_col] = mapping_df[cui_col].astype(str).str.strip()
mapping_df[term_col] = mapping_df[term_col].astype(str).str.strip()

# Keep only valid CUI rows
mapping_df = mapping_df[
    mapping_df[cui_col].str.match(r"^C\d{7}$", na=False)
].copy()

# Remove empty / invalid terms
mapping_df = mapping_df[
    mapping_df[term_col].notna()
    & (mapping_df[term_col].astype(str).str.strip() != "")
    & (mapping_df[term_col].astype(str).str.lower().str.strip() != "nan")
].copy()

# Drop duplicate CUIs while keeping first occurrence
mapping_df = mapping_df.drop_duplicates(subset=[cui_col], keep="first")

cui_to_term = dict(zip(mapping_df[cui_col], mapping_df[term_col]))

# Manual correction from previous experiment notes
cui_to_term["C0043262"] = "Wrist"

mapping_path = MAPPING_PATH

print("\nLoaded mappings:", len(cui_to_term))
print("Example C0040405:", cui_to_term.get("C0040405"))
print("Example C0043262:", cui_to_term.get("C0043262"))

missing_in_mapping = [c for c in cui_list if c not in cui_to_term]

print("\nCUIs in label vocab:", len(cui_list))
print("CUIs missing from mapping:", len(missing_in_mapping))
print("First missing:", missing_in_mapping[:20])

# Save a cleaned copy for reproducibility
clean_mapping_path = PROJECT_DIR / "cui_to_umls_terms_cleaned_for_exp3.csv"
mapping_df[[cui_col, term_col]].rename(
    columns={cui_col: "CUI", term_col: "UMLS_Term"}
).to_csv(clean_mapping_path, index=False)

print("\nSaved cleaned mapping to:", clean_mapping_path)

Using mapping file: /content/drive/MyDrive/Senior2_Medical_Captioning/cui_to_umls_terms.csv

Mapping shape: (2646, 5)
Mapping columns:
['CUI', 'name', 'semantic_types', 'status_code', 'error']


,CUI,name,semantic_types,status_code,error
0,C0000726,Abdomen,Body Location or Region,200,NaN
1,C0000741,Abducens nerve structure,"Body Part, Organ, or Organ Component",200,NaN
2,C0000833,Abscess,Disease or Syndrome,200,NaN
3,C0000846,Agenesis,Congenital Abnormality,200,NaN
4,C0000962,Bone structure of acetabulum,"Body Part, Organ, or Organ Component",200,NaN



Detected CUI column: CUI
Detected term column: name

Loaded mappings: 2639
Example C0040405: X-Ray Computed Tomography
Example C0043262: Wrist

CUIs in label vocab: 2646
CUIs missing from mapping: 7
First missing: ['C0206702', 'C0227130', 'C0241790', 'C0555895', 'C1134719', 'C1321581', 'C1458136']

Saved cleaned mapping to: /content/drive/MyDrive/Senior2_Medical_Captioning/cui_to_umls_terms_cleaned_for_exp3.csv


In [7]:
# ============================================================
# Text construction helpers
# ============================================================

def clean_cuis(x):
    """
    Convert a CUI string into a clean list.
    Supports semicolon, comma, whitespace-separated formats.
    """
    if pd.isna(x):
        return []

    s = str(x).strip()
    if s == "" or s.lower() in ["nan", "none", "null"]:
        return []

    # Normalize separators
    s = s.replace(",", ";")
    parts = []

    for chunk in s.split(";"):
        chunk = chunk.strip()
        if not chunk:
            continue

        # If chunk accidentally contains spaces between CUIs
        subparts = chunk.split()
        for sp in subparts:
            sp = sp.strip()
            if sp:
                parts.append(sp)

    # Keep order and remove duplicates
    seen = set()
    clean = []
    for c in parts:
        if c not in seen:
            seen.add(c)
            clean.append(c)

    return clean


def cui_to_readable_piece(cui, mapping):
    """
    Convert one CUI into '<CUI> term' format.
    """
    term = mapping.get(cui, None)

    if term is None or str(term).strip() == "":
        return f"<{cui}>"

    return f"<{cui}> {str(term).strip()}"


def make_cui_terms_input(cui_string, source_name, mapping):
    """
    Build T5 input using both CUI special tokens and readable UMLS terms.
    """
    cuis = clean_cuis(cui_string)

    if len(cuis) == 0:
        concept_text = "<NO_CONCEPTS>"
    else:
        pieces = [cui_to_readable_piece(cui, mapping) for cui in cuis]
        concept_text = "; ".join(pieces)

    return f"generate medical caption from {source_name} umls concepts: {concept_text}"


# Quick input examples
row0 = master_df.iloc[0]

print("GT input example:")
print(make_cui_terms_input(row0["gt_CUIs"], "ground-truth", cui_to_term))

print("\nPred micro input example:")
print(make_cui_terms_input(row0["pred_CUIs_micro"], "predicted", cui_to_term))

print("\nPred coverage input example:")
print(make_cui_terms_input(row0["pred_CUIs_coverage"], "predicted", cui_to_term))

GT input example:
generate medical caption from ground-truth umls concepts: <C0040405> X-Ray Computed Tomography

Pred micro input example:
generate medical caption from predicted umls concepts: <C0040405> X-Ray Computed Tomography

Pred coverage input example:
generate medical caption from predicted umls concepts: <C0040405> X-Ray Computed Tomography


In [8]:
# ============================================================
# Build train/valid dataframes for Experiment 3
# Training input: predicted micro CUIs + UMLS terms
# Target: reference caption
# ============================================================

df = master_df.copy()

# Remove invalid target captions
df["reference_caption"] = df["reference_caption"].astype(str)
df = df[
    df["reference_caption"].notna()
    & (df["reference_caption"].str.strip() != "")
    & (df["reference_caption"].str.lower().str.strip() != "nan")
].reset_index(drop=True)

df["input_text"] = df["pred_CUIs_micro"].apply(
    lambda x: make_cui_terms_input(x, "predicted", cui_to_term)
)

df["target_text"] = df["reference_caption"].astype(str).str.strip()

train_full_df = df[df["split"] == "train"][["ID", "input_text", "target_text"]].reset_index(drop=True)
valid_full_df = df[df["split"] == "valid"][["ID", "input_text", "target_text"]].reset_index(drop=True)

if len(train_full_df) == 0:
    raise ValueError("train_full_df is empty.")

if len(valid_full_df) == 0:
    raise ValueError("valid_full_df is empty.")

# Official validation sample
valid_n = min(VALID_SAMPLE_SIZE, len(valid_full_df))
valid_eval_df = valid_full_df.sample(n=valid_n, random_state=SEED).reset_index(drop=True)

# Optional debug mode
if FAST_DEBUG:
    train_n = min(TRAIN_DEBUG_SIZE, len(train_full_df))
    valid_debug_n = min(VALID_DEBUG_SIZE, len(valid_eval_df))

    train_full_df = train_full_df.sample(n=train_n, random_state=SEED).reset_index(drop=True)
    valid_eval_df = valid_eval_df.sample(n=valid_debug_n, random_state=SEED).reset_index(drop=True)

print("Train full:", train_full_df.shape)
print("Valid full:", valid_full_df.shape)
print("Valid eval:", valid_eval_df.shape)

print("\nExample input:")
print(train_full_df.iloc[0]["input_text"])

print("\nExample target:")
print(train_full_df.iloc[0]["target_text"])

print("\nEmpty input_text:", (train_full_df["input_text"].str.strip() == "").sum())
print("Empty target_text:", (train_full_df["target_text"].str.strip() == "").sum())

Train full: (97364, 3)
Valid full: (19240, 3)
Valid eval: (3000, 3)

Example input:
generate medical caption from predicted umls concepts: <C0040405> X-Ray Computed Tomography

Example target:
Head CT demonstrating left parotiditis.

Empty input_text: 0
Empty target_text: 0


In [9]:
# ============================================================
# Load tokenizer and model
# Prefer starting from Experiment 2 checkpoint if available
# ============================================================

MODEL_NAME = "t5-base"

# Search for previous GT + UMLS terms checkpoint
manual_candidates = [
    PROJECT_DIR / "models" / "gt_cui_terms_t5_base_full" / "checkpoint-3043",
    PROJECT_DIR / "models" / "gt_cui_umls_terms_t5_base_full" / "checkpoint-3043",
    PROJECT_DIR / "models" / "gt_umls_terms_t5_base_full" / "checkpoint-3043",
    PROJECT_DIR / "models" / "cui_terms_t5_base_full" / "checkpoint-3043",
    PROJECT_DIR / "caption_generator_datasets_umls_terms" / "checkpoint-3043",
]

auto_candidates = []
for root in [
    PROJECT_DIR,
    PROJECT_DIR / "models",
    PROJECT_DIR / "caption_generator_datasets_umls_terms",
]:
    if root.exists():
        auto_candidates.extend(list(root.rglob("checkpoint-3043")))
        auto_candidates.extend(list(root.rglob("best_model")))

all_candidates = manual_candidates + auto_candidates

valid_model_candidates = []
for p in all_candidates:
    if p.exists() and (p / "config.json").exists():
        valid_model_candidates.append(p)

BASE_CHECKPOINT = None

# Prefer checkpoint-3043 specifically
ckpt_3043 = [p for p in valid_model_candidates if p.name == "checkpoint-3043"]
if len(ckpt_3043) > 0:
    BASE_CHECKPOINT = ckpt_3043[0]
elif len(valid_model_candidates) > 0:
    BASE_CHECKPOINT = valid_model_candidates[0]

print("Model checkpoint candidates:")
for p in valid_model_candidates[:20]:
    print(" -", p)

if BASE_CHECKPOINT is not None:
    model_source = str(BASE_CHECKPOINT)
    print("\nStarting from previous checkpoint:", model_source)
else:
    model_source = MODEL_NAME
    print("\nNo previous checkpoint found. Starting from:", MODEL_NAME)

# Load base tokenizer and add CUI tokens in label_vocab order
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

special_tokens = [f"<{cui}>" for cui in cui_list] + ["<NO_CONCEPTS>"]
num_added = tokenizer.add_tokens(special_tokens)

print("Added tokens:", num_added)
print("Tokenizer size:", len(tokenizer))

model = AutoModelForSeq2SeqLM.from_pretrained(model_source)

old_vocab_size = model.get_input_embeddings().weight.shape[0]
new_vocab_size = len(tokenizer)

if old_vocab_size != new_vocab_size:
    print(f"Resizing token embeddings: {old_vocab_size} -> {new_vocab_size}")
    model.resize_token_embeddings(new_vocab_size)
else:
    print("Model embedding size already matches tokenizer size.")

# Important for stable seq2seq fine-tuning with gradient checkpointing
model.config.use_cache = False

print("Model loaded successfully.")

Model checkpoint candidates:
 - /content/drive/MyDrive/Senior2_Medical_Captioning/models/gt_cui_terms_t5_base_full/checkpoint-3043
 - /content/drive/MyDrive/Senior2_Medical_Captioning/models/gt_cui_terms_t5_base_full/checkpoint-3043
 - /content/drive/MyDrive/Senior2_Medical_Captioning/models/gt_cui_terms_t5_base_full/checkpoint-3043

Starting from previous checkpoint: /content/drive/MyDrive/Senior2_Medical_Captioning/models/gt_cui_terms_t5_base_full/checkpoint-3043


config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

Added tokens: 2647
Tokenizer size: 34747


Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

[transformers] The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Resizing token embeddings: 32128 -> 34747
Model loaded successfully.


In [10]:
# ============================================================
# Tokenization
# ============================================================

MAX_INPUT_LEN = 384
MAX_TARGET_LEN = 96

train_dataset = Dataset.from_pandas(train_full_df)
valid_dataset = Dataset.from_pandas(valid_eval_df)

def preprocess_function(batch):
    model_inputs = tokenizer(
        batch["input_text"],
        max_length=MAX_INPUT_LEN,
        truncation=True
    )

    labels = tokenizer(
        text_target=batch["target_text"],
        max_length=MAX_TARGET_LEN,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


tokenized_train = train_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=train_dataset.column_names
)

tokenized_valid = valid_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=valid_dataset.column_names
)

print(tokenized_train)
print(tokenized_valid)

# Quick length checks
sample_item = tokenized_train[0]
print("\nSample tokenized keys:", sample_item.keys())
print("Input length:", len(sample_item["input_ids"]))
print("Label length:", len(sample_item["labels"]))

Map:   0%|          | 0/97364 [00:00<?, ? examples/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 97364
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 3000
})

Sample tokenized keys: dict_keys(['input_ids', 'attention_mask', 'labels'])
Input length: 21
Label length: 14


In [11]:
# ============================================================
# Evaluation metrics
# ============================================================

import evaluate

rouge = evaluate.load("rouge")
bleu = evaluate.load("sacrebleu")

def compute_metrics(eval_pred):
    preds, labels = eval_pred

    if isinstance(preds, tuple):
        preds = preds[0]

    preds = np.asarray(preds)
    labels = np.asarray(labels)

    # If predictions are logits, convert to token ids
    if preds.ndim == 3:
        preds = np.argmax(preds, axis=-1)

    vocab_size = len(tokenizer)

    # Replace invalid prediction token ids
    preds = np.where(
        (preds < 0) | (preds >= vocab_size),
        tokenizer.pad_token_id,
        preds
    )

    # Replace ignored label ids
    labels = np.where(
        labels != -100,
        labels,
        tokenizer.pad_token_id
    )

    preds = preds.astype(np.int64)
    labels = labels.astype(np.int64)

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds = [p.strip() for p in decoded_preds]
    decoded_labels = [l.strip() for l in decoded_labels]

    rouge_result = rouge.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        use_stemmer=True
    )

    bleu_result = bleu.compute(
        predictions=decoded_preds,
        references=[[l] for l in decoded_labels]
    )

    return {
        "rouge1": rouge_result["rouge1"],
        "rouge2": rouge_result["rouge2"],
        "rougeL": rouge_result["rougeL"],
        "bleu": bleu_result["score"]
    }

In [12]:
# ============================================================
# Safety forward pass
# This checks that one mini-batch produces finite loss before training.
# ============================================================

from transformers import DataCollatorForSeq2Seq

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.train()

mini_features = [tokenized_train[i] for i in range(min(2, len(tokenized_train)))]
mini_batch = data_collator(mini_features)

mini_batch = {
    k: v.to(device) if hasattr(v, "to") else v
    for k, v in mini_batch.items()
}

with torch.no_grad():
    outputs = model(**mini_batch)
    test_loss = outputs.loss

print("Safety test loss:", float(test_loss.detach().cpu()))

if not torch.isfinite(test_loss):
    raise ValueError("Non-finite loss detected before training. Stop and inspect the data/model.")
else:
    print("Safety check passed: finite loss.")

Safety test loss: 3.60233211517334
Safety check passed: finite loss.


In [14]:
# ============================================================
# Safe 5-epoch training for Experiment 3
# Fixed version: automatically removes unsupported TrainingArguments
# ============================================================

import os
import gc
import inspect
import numpy as np
import pandas as pd
import torch

from transformers import (
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    EarlyStoppingCallback,
    TrainerCallback,
    DataCollatorForSeq2Seq
)
from transformers.trainer_utils import get_last_checkpoint

# ============================================================
# Clear GPU memory
# ============================================================

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Enable TF32 on A100 for speed/stability
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

# Precision policy:
# Use BF16 on A100 if available.
# Avoid FP16 because of previous NaN risk.
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
use_fp16 = False

print("use_bf16:", use_bf16)
print("use_fp16:", use_fp16)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# Important for gradient checkpointing
model.config.use_cache = False


# ============================================================
# Callback: stop if NaN/Inf appears
# ============================================================

class StopOnNonFiniteCallback(TrainerCallback):
    """
    Stop training if NaN/Inf appears in logs or evaluation metrics.
    """
    def _check_values(self, values, control, stage):
        if values is None:
            return control

        for key, value in values.items():
            if isinstance(value, (int, float, np.floating)):
                if not np.isfinite(value):
                    print(f"\n[SAFETY STOP] Non-finite value detected during {stage}: {key}={value}")
                    control.should_save = True
                    control.should_training_stop = True
                    return control

        return control

    def on_log(self, args, state, control, logs=None, **kwargs):
        return self._check_values(logs, control, "logging")

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        return self._check_values(metrics, control, "evaluation")


# ============================================================
# Build TrainingArguments safely
# ============================================================

args_dict = dict(
    output_dir=str(OUTPUT_DIR),

    # Train up to 5 epochs
    num_train_epochs=5,

    # Evaluation and saving
    eval_strategy="epoch",
    save_strategy="epoch",

    # Safer learning rate
    learning_rate=2e-5,
    warmup_ratio=0.08,
    lr_scheduler_type="linear",

    # Effective batch size = 8 * 4 = 32
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,

    # Stability controls
    max_grad_norm=0.5,
    weight_decay=0.01,
    adam_epsilon=1e-6,
    label_smoothing_factor=0.05,

    # Precision
    bf16=use_bf16,
    fp16=use_fp16,

    # Generation during evaluation
    predict_with_generate=True,
    generation_max_length=96,
    generation_num_beams=2,

    # Reduce GPU memory spikes during evaluation
    eval_accumulation_steps=8,

    # Logging/saving
    logging_steps=100,
    save_total_limit=3,
    report_to="none",
    logging_nan_inf_filter=True,

    # Best checkpoint selection
    load_best_model_at_end=True,
    metric_for_best_model="rougeL",
    greater_is_better=True,

    # Memory saving
    gradient_checkpointing=True,

    # Reproducibility
    seed=SEED,
    data_seed=SEED,
)

# Add save_safetensors only if supported by this transformers version
supported_args = set(inspect.signature(Seq2SeqTrainingArguments.__init__).parameters.keys())

# Handle eval_strategy vs evaluation_strategy compatibility
if "eval_strategy" not in supported_args and "evaluation_strategy" in supported_args:
    args_dict["evaluation_strategy"] = args_dict.pop("eval_strategy")
elif "eval_strategy" in supported_args:
    pass
else:
    # Very old versions may not support either name
    args_dict.pop("eval_strategy", None)

# Remove unsupported arguments automatically
filtered_args = {}
removed_args = []

for key, value in args_dict.items():
    if key in supported_args:
        filtered_args[key] = value
    else:
        removed_args.append(key)

print("\nUnsupported TrainingArguments removed automatically:")
print(removed_args)

training_args = Seq2SeqTrainingArguments(**filtered_args)

print("\nTrainingArguments created successfully.")


# ============================================================
# Trainer
# ============================================================

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_valid,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[
        StopOnNonFiniteCallback(),
        EarlyStoppingCallback(
            early_stopping_patience=2,
            early_stopping_threshold=0.0005
        )
    ]
)

last_checkpoint = get_last_checkpoint(str(OUTPUT_DIR))
print("\nLast checkpoint:", last_checkpoint)


# ============================================================
# Train
# ============================================================

if last_checkpoint is not None:
    print("Resuming from checkpoint:", last_checkpoint)
    train_result = trainer.train(resume_from_checkpoint=last_checkpoint)
else:
    print("Starting fresh safe 5-epoch training.")
    train_result = trainer.train()


# ============================================================
# Save best model
# ============================================================

BEST_MODEL_DIR = OUTPUT_DIR / "best_model"
trainer.save_model(str(BEST_MODEL_DIR))
tokenizer.save_pretrained(str(BEST_MODEL_DIR))

print("\nSaved best model to:", BEST_MODEL_DIR)


# ============================================================
# Save training metrics and history
# ============================================================

train_metrics = train_result.metrics

trainer.log_metrics("train", train_metrics)
trainer.save_metrics("train", train_metrics)
trainer.save_state()

history_df = pd.DataFrame(trainer.state.log_history)
history_path = OUTPUT_DIR / "training_log_history.csv"
history_df.to_csv(history_path, index=False)

print("\nTraining completed.")
print("Training metrics:")
print(train_metrics)

print("\nSaved training history to:", history_path)

display(history_df.tail(20))

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


use_bf16: True
use_fp16: False
GPU: NVIDIA A100-SXM4-80GB

Unsupported TrainingArguments removed automatically:
[]

TrainingArguments created successfully.

Last checkpoint: None
Starting fresh safe 5-epoch training.


Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Bleu
1,14.083629,3.340528,0.208327,0.063280,0.180706,2.276206


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Bleu
1,14.083629,3.340528,0.208327,0.063280,0.180706,2.276206
2,13.675347,3.268091,0.191331,0.060759,0.166103,1.945277
3,13.591798,3.233163,0.191331,0.060574,0.166370,2.327277


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Saved best model to: /content/drive/MyDrive/Senior2_Medical_Captioning/models/exp3_pred_micro_terms_t5_base_safe5/best_model
***** train metrics *****
  epoch                    =        3.0
  total_flos               =  8955506GF
  train_loss               =    14.0806
  train_runtime            = 3:04:53.09
  train_samples_per_second =     43.885
  train_steps_per_second   =      1.372

Training completed.
Training metrics:
{'train_runtime': 11093.0968, 'train_samples_per_second': 43.885, 'train_steps_per_second': 1.372, 'total_flos': 9615901810360320.0, 'train_loss': 14.080610332307241, 'epoch': 3.0}

Saved training history to: /content/drive/MyDrive/Senior2_Medical_Captioning/models/exp3_pred_micro_terms_t5_base_safe5/training_log_history.csv


,loss,grad_norm,learning_rate,epoch,step,eval_loss,eval_rouge1,eval_rouge2,eval_rougeL,eval_bleu,eval_runtime,eval_samples_per_second,eval_steps_per_second,train_runtime,train_samples_per_second,train_steps_per_second,total_flos,train_loss
75,13.633824,3.100792,0.000011,2.431846,7400,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
76,13.517924,2.697896,0.000011,2.464711,7500,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
77,13.630087,2.603356,0.000011,2.497576,7600,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
78,13.600726,2.846745,0.000011,2.530441,7700,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
79,13.588622,2.884142,0.000011,2.563306,7800,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
80,13.609669,3.244770,0.000010,2.596171,7900,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
81,13.520743,3.021489,0.000010,2.629036,8000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
82,13.616000,3.272049,0.000010,2.661901,8100,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
83,13.544529,2.674358,0.000010,2.694766,8200,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
84,13.456357,3.114314,0.000010,2.727631,8300,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [17]:
# ============================================================
# Build evaluation variants using the same validation sample IDs
# ============================================================

valid_ids = valid_eval_df["ID"].astype(str).tolist()

eval_base_df = master_df[master_df["ID"].astype(str).isin(valid_ids)].copy()

# Preserve the same order as valid_eval_df
eval_base_df["ID_order"] = pd.Categorical(
    eval_base_df["ID"].astype(str),
    categories=valid_ids,
    ordered=True
)

eval_base_df = eval_base_df.sort_values("ID_order").drop(columns=["ID_order"]).reset_index(drop=True)

print("Eval base shape:", eval_base_df.shape)
print("Expected valid ids:", len(valid_ids))

if len(eval_base_df) != len(valid_ids):
    print("Warning: eval_base_df length differs from valid_ids length.")
    print("This may happen if IDs are duplicated or missing.")


def build_eval_variant(base_df, cui_col, source_name):
    temp = base_df.copy()

    temp["input_text"] = temp[cui_col].apply(
        lambda x: make_cui_terms_input(x, source_name, cui_to_term)
    )

    temp["target_text"] = temp["reference_caption"].astype(str).str.strip()

    return temp[["ID", "input_text", "target_text"]].reset_index(drop=True)


valid_gt_eval = build_eval_variant(
    eval_base_df,
    "gt_CUIs",
    "ground-truth"
)

valid_micro_eval = build_eval_variant(
    eval_base_df,
    "pred_CUIs_micro",
    "predicted"
)

valid_coverage_eval = build_eval_variant(
    eval_base_df,
    "pred_CUIs_coverage",
    "predicted"
)

print("GT eval:", valid_gt_eval.shape)
print("Pred micro eval:", valid_micro_eval.shape)
print("Pred coverage eval:", valid_coverage_eval.shape)

print("\nGT example:")
print(valid_gt_eval.iloc[0]["input_text"])

print("\nPred micro example:")
print(valid_micro_eval.iloc[0]["input_text"])

print("\nPred coverage example:")
print(valid_coverage_eval.iloc[0]["input_text"])

Eval base shape: (3000, 20)
Expected valid ids: 3000
GT eval: (3000, 3)
Pred micro eval: (3000, 3)
Pred coverage eval: (3000, 3)

GT example:
generate medical caption from ground-truth umls concepts: <C0041618> Ultrasonography; <C0182400> Probes; <C0042449> Veins

Pred micro example:
generate medical caption from predicted umls concepts: <C0041618> Ultrasonography

Pred coverage example:
generate medical caption from predicted umls concepts: <C0041618> Ultrasonography


In [18]:
# ============================================================
# Tokenize evaluation variants
# ============================================================

def tokenize_eval_df(eval_df):
    dataset = Dataset.from_pandas(eval_df)

    tokenized = dataset.map(
        preprocess_function,
        batched=True,
        remove_columns=dataset.column_names
    )

    return tokenized


tokenized_valid_gt = tokenize_eval_df(valid_gt_eval)
tokenized_valid_micro = tokenize_eval_df(valid_micro_eval)
tokenized_valid_coverage = tokenize_eval_df(valid_coverage_eval)

print(tokenized_valid_gt)
print(tokenized_valid_micro)
print(tokenized_valid_coverage)

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 3000
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 3000
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 3000
})


In [19]:
# ============================================================
# Final evaluation of Experiment 3
# ============================================================

gt_metrics = trainer.evaluate(
    eval_dataset=tokenized_valid_gt,
    metric_key_prefix="gt"
)

micro_metrics = trainer.evaluate(
    eval_dataset=tokenized_valid_micro,
    metric_key_prefix="pred_micro"
)

coverage_metrics = trainer.evaluate(
    eval_dataset=tokenized_valid_coverage,
    metric_key_prefix="pred_coverage"
)

comparison = pd.DataFrame([
    {
        "model": "Exp3 Pred-micro fine-tuned safe5",
        "setting": "GT concepts + UMLS terms",
        "loss": gt_metrics.get("gt_loss"),
        "rouge1": gt_metrics.get("gt_rouge1"),
        "rouge2": gt_metrics.get("gt_rouge2"),
        "rougeL": gt_metrics.get("gt_rougeL"),
        "bleu": gt_metrics.get("gt_bleu"),
    },
    {
        "model": "Exp3 Pred-micro fine-tuned safe5",
        "setting": "Pred micro concepts + UMLS terms",
        "loss": micro_metrics.get("pred_micro_loss"),
        "rouge1": micro_metrics.get("pred_micro_rouge1"),
        "rouge2": micro_metrics.get("pred_micro_rouge2"),
        "rougeL": micro_metrics.get("pred_micro_rougeL"),
        "bleu": micro_metrics.get("pred_micro_bleu"),
    },
    {
        "model": "Exp3 Pred-micro fine-tuned safe5",
        "setting": "Pred coverage concepts + UMLS terms",
        "loss": coverage_metrics.get("pred_coverage_loss"),
        "rouge1": coverage_metrics.get("pred_coverage_rouge1"),
        "rouge2": coverage_metrics.get("pred_coverage_rouge2"),
        "rougeL": coverage_metrics.get("pred_coverage_rougeL"),
        "bleu": coverage_metrics.get("pred_coverage_bleu"),
    },
])

display(comparison)

comparison_path = OUTPUT_DIR / "exp3_safe5_gt_vs_pred_eval_comparison_valid3000.csv"
comparison.to_csv(comparison_path, index=False)

print("Saved comparison to:", comparison_path)

[transformers] early stopping required metric_for_best_model, but did not find eval_rougeL so early stopping is disabled


Training Loss,Validation Loss,Epoch,Rouge1,Rouge2,Rougel,Bleu
13.591798,3.113310,3,0.288020,0.106795,0.247639,5.020259


[transformers] early stopping required metric_for_best_model, but did not find eval_rougeL so early stopping is disabled


Training Loss,Validation Loss,Epoch,Micro Loss,Micro Rouge1,Micro Rouge2,Micro Rougel,Micro Bleu
13.591798,No log,3,3.340528,0.208327,0.063280,0.180706,2.276206


[transformers] early stopping required metric_for_best_model, but did not find eval_rougeL so early stopping is disabled


Training Loss,Validation Loss,Epoch,Coverage Loss,Coverage Rouge1,Coverage Rouge2,Coverage Rougel,Coverage Bleu
13.591798,No log,3,3.337225,0.211430,0.065665,0.183133,2.299728


,model,setting,loss,rouge1,rouge2,rougeL,bleu
0,Exp3 Pred-micro fine-tuned safe5,GT concepts + UMLS terms,3.113310,0.288020,0.106795,0.247639,5.020259
1,Exp3 Pred-micro fine-tuned safe5,Pred micro concepts + UMLS terms,3.340528,0.208327,0.063280,0.180706,2.276206
2,Exp3 Pred-micro fine-tuned safe5,Pred coverage concepts + UMLS terms,3.337225,0.211430,0.065665,0.183133,2.299728


Saved comparison to: /content/drive/MyDrive/Senior2_Medical_Captioning/models/exp3_pred_micro_terms_t5_base_safe5/exp3_safe5_gt_vs_pred_eval_comparison_valid3000.csv


In [20]:
# ============================================================
# Compare previous GT-trained model results with Exp3 results
# These previous numbers are from the completed experiments in the handoff.
# ============================================================

previous_results = pd.DataFrame([
    {
        "model": "Exp2 GT-trained",
        "setting": "GT concepts + UMLS terms",
        "loss": 2.911293,
        "rouge1": 0.276256,
        "rouge2": 0.112285,
        "rougeL": 0.245186,
        "bleu": 5.134103,
    },
    {
        "model": "Exp2 GT-trained",
        "setting": "Pred micro concepts + UMLS terms",
        "loss": 3.370372,
        "rouge1": 0.145410,
        "rouge2": 0.043019,
        "rougeL": 0.130601,
        "bleu": 1.209188,
    },
    {
        "model": "Exp2 GT-trained",
        "setting": "Pred coverage concepts + UMLS terms",
        "loss": 3.369131,
        "rouge1": 0.144617,
        "rouge2": 0.044812,
        "rougeL": 0.129142,
        "bleu": 1.220970,
    },
])

full_comparison = pd.concat(
    [previous_results, comparison],
    ignore_index=True
)

# Add improvement columns relative to old practical baseline where possible
old_micro_rougeL = 0.130601
old_micro_bleu = 1.209188

full_comparison["delta_vs_old_pred_micro_rougeL"] = full_comparison["rougeL"] - old_micro_rougeL
full_comparison["delta_vs_old_pred_micro_bleu"] = full_comparison["bleu"] - old_micro_bleu

display(full_comparison)

full_comparison_path = OUTPUT_DIR / "exp2_vs_exp3_safe5_full_comparison_valid3000.csv"
full_comparison.to_csv(full_comparison_path, index=False)

print("Saved full comparison to:", full_comparison_path)

,model,setting,loss,rouge1,rouge2,rougeL,bleu,delta_vs_old_pred_micro_rougeL,delta_vs_old_pred_micro_bleu
0,Exp2 GT-trained,GT concepts + UMLS terms,2.911293,0.276256,0.112285,0.245186,5.134103,0.114585,3.924915
1,Exp2 GT-trained,Pred micro concepts + UMLS terms,3.370372,0.145410,0.043019,0.130601,1.209188,0.000000,0.000000
2,Exp2 GT-trained,Pred coverage concepts + UMLS terms,3.369131,0.144617,0.044812,0.129142,1.220970,-0.001459,0.011782
3,Exp3 Pred-micro fine-tuned safe5,GT concepts + UMLS terms,3.113310,0.288020,0.106795,0.247639,5.020259,0.117038,3.811071
4,Exp3 Pred-micro fine-tuned safe5,Pred micro concepts + UMLS terms,3.340528,0.208327,0.063280,0.180706,2.276206,0.050105,1.067018
5,Exp3 Pred-micro fine-tuned safe5,Pred coverage concepts + UMLS terms,3.337225,0.211430,0.065665,0.183133,2.299728,0.052532,1.090540


Saved full comparison to: /content/drive/MyDrive/Senior2_Medical_Captioning/models/exp3_pred_micro_terms_t5_base_safe5/exp2_vs_exp3_safe5_full_comparison_valid3000.csv


In [21]:
# ============================================================
# Qualitative sample generation
# ============================================================

device = "cuda" if torch.cuda.is_available() else "cpu"

model.to(device)
model.eval()

sample_n = min(10, len(eval_base_df))
sample_df = eval_base_df.sample(n=sample_n, random_state=11).reset_index(drop=True)

qualitative_rows = []

def generate_caption_from_input(input_text, num_beams=4):
    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_INPUT_LEN
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=96,
            num_beams=num_beams,
            early_stopping=True
        )

    pred = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return pred.strip()


for _, row in sample_df.iterrows():
    sample_id = row["ID"]
    target = row["reference_caption"]

    gt_input = make_cui_terms_input(row["gt_CUIs"], "ground-truth", cui_to_term)
    micro_input = make_cui_terms_input(row["pred_CUIs_micro"], "predicted", cui_to_term)
    coverage_input = make_cui_terms_input(row["pred_CUIs_coverage"], "predicted", cui_to_term)

    gt_pred = generate_caption_from_input(gt_input)
    micro_pred = generate_caption_from_input(micro_input)
    coverage_pred = generate_caption_from_input(coverage_input)

    qualitative_rows.append({
        "ID": sample_id,
        "target": target,
        "gt_CUIs": row["gt_CUIs"],
        "pred_CUIs_micro": row["pred_CUIs_micro"],
        "pred_CUIs_coverage": row["pred_CUIs_coverage"],
        "gt_input": gt_input,
        "micro_input": micro_input,
        "coverage_input": coverage_input,
        "gt_prediction": gt_pred,
        "micro_prediction": micro_pred,
        "coverage_prediction": coverage_pred,
    })

    print("=" * 120)
    print("ID:", sample_id)

    print("\nTARGET:")
    print(target)

    print("\nGT CUIs:")
    print(row["gt_CUIs"])

    print("\nPRED MICRO CUIs:")
    print(row["pred_CUIs_micro"])

    print("\nPRED COVERAGE CUIs:")
    print(row["pred_CUIs_coverage"])

    print("\nGT PREDICTION:")
    print(gt_pred)

    print("\nMICRO PREDICTION:")
    print(micro_pred)

    print("\nCOVERAGE PREDICTION:")
    print(coverage_pred)

qualitative_df = pd.DataFrame(qualitative_rows)

qualitative_path = OUTPUT_DIR / "exp3_safe5_qualitative_samples.csv"
qualitative_df.to_csv(qualitative_path, index=False)

print("\nSaved qualitative samples to:", qualitative_path)

display(qualitative_df[[
    "ID",
    "target",
    "gt_prediction",
    "micro_prediction",
    "coverage_prediction"
]])

ID: ImageCLEFmedical_Caption_2026_valid_4390

TARGET:
Example of an anterior–posterior radiograph of the pelvis with calculation of the migration percentage (MP): MP = A/B × 100. A represents the portion of ossified femoral head laying lateral to Perkin's line (vertical line drawn through the lateral acetabular margin and perpendicular to Hilgenreiner's line, which passes through the superior aspect of the triradiate cartilage). B represents the whole ossified femoral head.

GT CUIs:
C1306645;C0023216;C0030797;C0015813;C0007301

PRED MICRO CUIs:
C0030797;C1306645

PRED COVERAGE CUIs:
C0015813;C0030797;C1306645

GT PREDICTION:
Anteroposterior radiograph of the femoral head.

MICRO PREDICTION:
Anteroposterior radiograph of the pelvis showing a large obstructive lesion in the right lower lobe of the pelvis.

COVERAGE PREDICTION:
Anteroposterior radiograph of the femoral head.
ID: ImageCLEFmedical_Caption_2026_valid_15145

TARGET:
Coronal view of contrast-enhanced magnetic resonance imagin

,ID,target,gt_prediction,micro_prediction,coverage_prediction
0,ImageCLEFmedical_Caption_2026_valid_4390,Example of an anterior–posterior radiograph of...,Anteroposterior radiograph of the femoral head.,Anteroposterior radiograph of the pelvis showi...,Anteroposterior radiograph of the femoral head.
1,ImageCLEFmedical_Caption_2026_valid_15145,Coronal view of contrast-enhanced magnetic res...,Axial T1-weighted MRI of the saphenous vein sh...,Axial T1-weighted MRI of the thoracic spine sh...,Axial T1-weighted MRI of the thoracic spine sh...
2,ImageCLEFmedical_Caption_2026_valid_5073,Radiographic check-up after the provisional re...,Panoramic radiograph of the patient.,Panoramic radiograph of the patient.,Panoramic radiograph of the patient.
3,ImageCLEFmedical_Caption_2026_valid_2299,A 57-year-old woman underwent pelvic ultrasoun...,Transesophageal echocardiography of the pelvis...,Transesophageal echocardiography (TEE) showing...,Transesophageal echocardiography (TEE) showing...
4,ImageCLEFmedical_Caption_2026_valid_15195,Image showing right-sided bipolar hemiarthropl...,Anteroposterior radiograph of the femur.,Anteroposterior X-ray of the femur.,Anteroposterior X-ray of the femur.
5,ImageCLEFmedical_Caption_2026_valid_19195,Coronal T2 weighted magnetic resonance image o...,Axial T1-weighted MRI of the left ankle demons...,Axial T1-weighted MRI of the thoracic spine sh...,Axial T1-weighted MRI of the thoracic spine sh...
6,ImageCLEFmedical_Caption_2026_valid_265,Pneumomediastinum in the superior mediastinum ...,Computed tomography (CT) scan of the abdomen a...,Computed tomography (CT) scan of the abdomen a...,Chest computed tomography (CT) scan showing a ...
7,ImageCLEFmedical_Caption_2026_valid_11915,Day 2 transvaginal sonography of the patient's...,Transesophageal echocardiography showing a lar...,Transesophageal echocardiography (TEE) showing...,Transesophageal echocardiography (TEE) showing...
8,ImageCLEFmedical_Caption_2026_valid_9689,Repeat chest X-ray after one week. In comparis...,Chest X-ray showing a right-sided pulmonary em...,Chest X-ray showing a large right-sided pleura...,Chest X-ray showing a large right-sided pleura...
9,ImageCLEFmedical_Caption_2026_valid_10849,Post EM Titration (admit 2).Marked left lower ...,Chest X-ray of the patient showing a large rig...,Chest X-ray showing a large right-sided pleura...,Chest X-ray showing a large right-sided pleura...


In [22]:
# ============================================================
# Save run metadata
# ============================================================

run_summary = {
    "experiment": "Experiment 3 - Predicted-concept fine-tuning",
    "variant": "pred_micro_terms_t5_base_safe5",
    "project_dir": str(PROJECT_DIR),
    "master_path": str(MASTER_PATH),
    "label_vocab_path": str(LABEL_VOCAB_PATH),
    "mapping_path": str(mapping_path),
    "output_dir": str(OUTPUT_DIR),
    "base_model_or_checkpoint": model_source,
    "model_name": MODEL_NAME,
    "num_cuis": len(cui_list),
    "train_size": int(len(train_full_df)),
    "valid_eval_size": int(len(valid_eval_df)),
    "max_input_len": MAX_INPUT_LEN,
    "max_target_len": MAX_TARGET_LEN,
    "training_epochs_max": 5,
    "learning_rate": 2e-5,
    "warmup_ratio": 0.08,
    "effective_batch_size": 8 * 4,
    "precision": "bf16" if use_bf16 else "fp32",
    "fp16_used": False,
    "best_model_dir": str(BEST_MODEL_DIR),
    "comparison_csv": str(comparison_path),
    "full_comparison_csv": str(full_comparison_path),
    "qualitative_csv": str(qualitative_path),
}

summary_path = OUTPUT_DIR / "exp3_safe5_run_summary.json"

with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(run_summary, f, indent=2, ensure_ascii=False)

print("Saved run summary to:", summary_path)
print(json.dumps(run_summary, indent=2, ensure_ascii=False))

Saved run summary to: /content/drive/MyDrive/Senior2_Medical_Captioning/models/exp3_pred_micro_terms_t5_base_safe5/exp3_safe5_run_summary.json
{
  "experiment": "Experiment 3 - Predicted-concept fine-tuning",
  "variant": "pred_micro_terms_t5_base_safe5",
  "project_dir": "/content/drive/MyDrive/Senior2_Medical_Captioning",
  "master_path": "/content/drive/MyDrive/Senior2_Medical_Captioning/master_caption_concepts_scored.csv",
  "label_vocab_path": "/content/drive/MyDrive/ImageCLEF/artifacts_consultive_swin/label_vocab.json",
  "mapping_path": "/content/drive/MyDrive/Senior2_Medical_Captioning/cui_to_umls_terms.csv",
  "output_dir": "/content/drive/MyDrive/Senior2_Medical_Captioning/models/exp3_pred_micro_terms_t5_base_safe5",
  "base_model_or_checkpoint": "/content/drive/MyDrive/Senior2_Medical_Captioning/models/gt_cui_terms_t5_base_full/checkpoint-3043",
  "model_name": "t5-base",
  "num_cuis": 2646,
  "train_size": 97364,
  "valid_eval_size": 3000,
  "max_input_len": 384,
  "max_tar